# Feature Engineering and Selection
This notebook focuses on creating time-series features such as lag variables, rolling window statistics, and seasonal indicators to improve our prediction models.


In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = '../data'


## Loading Data & Creating Base Time Features


In [ ]:
weather = pd.read_csv(os.path.join(DATA_DIR, 'weather.csv'))
pollution = pd.read_csv(os.path.join(DATA_DIR, 'pollution.csv'))
df = pd.merge(weather, pollution, on=['Date', 'City'])
df['Date'] = pd.to_datetime(df['Date'])

# Base Date features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
print(df[['Date', 'Month', 'DayOfWeek', 'IsWeekend']].head())


## Generating Lag and Rolling Window Features


In [ ]:
# 1-day and 7-day Lags for AQI
df['AQI_Lag_1'] = df.groupby('City')['AQI'].shift(1)
df['AQI_Lag_7'] = df.groupby('City')['AQI'].shift(7)

# 7-day Rolling Averages
df['AQI_7Day_Mean'] = df.groupby('City')['AQI'].transform(lambda x: x.shift(1).rolling(7).mean())
df['AQI_7Day_Std'] = df.groupby('City')['AQI'].transform(lambda x: x.shift(1).rolling(7).std())

# Fill or drop NaNs due to shifting
df.dropna(inplace=True)
print('Features summary stats:\n', df[['AQI', 'AQI_Lag_1', 'AQI_7Day_Mean', 'AQI_7Day_Std']].describe())


## Feature Correlation Heatmap


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(df[['AQI', 'Temperature', 'Humidity', 'Wind_Speed', 'AQI_Lag_1', 'AQI_7Day_Mean']].corr(), annot=True, cmap='viridis')
plt.title('Engineered Features Correlation Analysis')
plt.show()
